# Notebook 1: GARCH(1,1) Estimation
**Project:** GARCH & Volatility Forecasting
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from arch import arch_model
import statsmodels.api as sm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy = returns['SPY'] * 100
spy_raw = returns['SPY']
print(f'Loaded {len(spy):,} obs | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 obs | 2015-01-01 to 2024-12-31


## 1. Fit GARCH(1,1)

In [2]:
from garch_models import fit_garch, garch_forecast
result = fit_garch(spy_raw)
params = result['params']
print(f"GARCH(1,1) Parameters:")
print(f"  omega       : {params['omega']:.6f}")
print(f"  alpha[1]    : {params['alpha[1]']:.6f}")
print(f"  beta[1]     : {params['beta[1]']:.6f}")
print(f"  persistence : {result['persistence']:.6f}")
print(f"  AIC         : {result['aic']}")
print(f"  BIC         : {result['bic']}")

GARCH(1,1) Parameters:
  omega       : 0.149078
  alpha[1]    : 0.060543
  beta[1]     : 0.807862
  persistence : 0.868405
  AIC         : 7692.88
  BIC         : 7716.34


## 2. Conditional Volatility

In [3]:
cond_vol = result['cond_vol']
fig, axes = plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
ax1.fill_between(spy.index,spy,0,where=spy<0,color='#E24B4A',alpha=0.5,label='Negative')
ax1.fill_between(spy.index,spy,0,where=spy>=0,color='#1D9E75',alpha=0.4,label='Positive')
ax1.set_title('SPY Daily Returns (%)'); ax1.legend(fontsize=9)
ax2=axes[1]
ax2.plot(cond_vol.index,cond_vol*100,color='#185FA5',linewidth=1.2,label='GARCH(1,1) Cond. Vol')
ax2.fill_between(cond_vol.index,cond_vol*100,alpha=0.12,color='#185FA5')
ax2.set_title('GARCH(1,1) Conditional Volatility'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/01_garch11_conditional_vol.png',dpi=150,bbox_inches='tight')
plt.show(); print('Chart saved.')

Chart saved.


## 3. 22-Day Forecast

In [4]:
fc = garch_forecast(result, horizon=22)
print('22-Day Volatility Forecast:')
print(fc.to_string())
fc.to_csv('../results/garch11_forecast.csv')

22-Day Volatility Forecast:
         forecast_vol_daily  forecast_vol_annual
horizon                                         
1                  0.010102             0.160371
2                  0.010175             0.161528
3                  0.010238             0.162525
4                  0.010292             0.163387
5                  0.010339             0.164131
6                  0.010380             0.164775
7                  0.010415             0.165332
8                  0.010445             0.165814
9                  0.010472             0.166232
10                 0.010494             0.166593
11                 0.010514             0.166907
12                 0.010531             0.167179
13                 0.010546             0.167414
14                 0.010559             0.167619
15                 0.010570             0.167796
16                 0.010580             0.167950
17                 0.010588             0.168083
18                 0.010596             0

## 4. Key Findings

In [5]:
p = result['persistence']
print("="*50)
print("  GARCH(1,1) KEY FINDINGS")
print("="*50)
print(f"  Persistence (alpha+beta): {p:.4f}")
if p > 0.95: print("  High persistence — vol shocks decay slowly")
elif p > 0.9: print("  Moderate persistence — typical for equities")
else: print("  Lower persistence — faster mean reversion")
print("="*50)

  GARCH(1,1) KEY FINDINGS
  Persistence (alpha+beta): 0.8684
  Lower persistence — faster mean reversion
